<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Python Foundations: Catch-up Practice</div>
</div>

#### One idea at a time: arrays, the copy trap, slicing, masks, loops, functions, plots and just enough pandas.
***

## What this notebook is for

The labs and exercises use a small set of Python ideas again and again. If any of them feel shaky, this notebook practises them **one at a time**: a short example, then a two-line exercise with a **Check**, before anything is combined.

Do it in any order, a section at a time: 10–15 minutes each. You don't need to finish it in one go. If a section already feels easy, skip it.

| Section | Idea | Where you'll need it |
| --- | --- | --- |
| 1 | Lists and NumPy arrays | everywhere |
| 2 | The copy trap | every solver and loop |
| 3 | Indexing and slicing | differences, neighbours, parts of a record |
| 4 | Boolean masks | selecting, cleaning, counting |
| 5 | Loops that fill an array | Euler, RK4, any step-by-step method |
| 6 | Functions | anything you do more than once |
| 7 | Functions as inputs | solvers, `curve_fit`, root finders |
| 8 | Plots that say something | every figure you submit |
| 9 | Just enough pandas | tables with one row per person or group |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## <span style="color:#B5121B">1. Lists and NumPy arrays</span>

A Python **list** holds anything; a NumPy **array** holds numbers and does arithmetic on all of them at once.

In [ ]:
lst = [1.0, 2.0, 3.0]
arr = np.array([1.0, 2.0, 3.0])
print(lst * 2)        # a list repeats itself
print(arr * 2)        # an array multiplies every element
print(len(arr), arr.shape, arr.sum(), arr.mean())

**1.1** Make an array `t` of the times 0, 0.5, 1.0, …, 5.0 with `np.arange` (careful: the end point is *not* included), and an array `y = t**2`.

In [ ]:
# 1.1

**Check:** 11 values; the last time is 5.0 and the last `y` is 25.0.

**1.2** `np.linspace(a, b, n)` gives `n` values *including* both ends. Make 101 values from 0 to 1. What is the spacing?

In [ ]:
# 1.2

**Check:** 0.01.

## <span style="color:#B5121B">2. The copy trap</span>

`b = a` does **not** copy an array: `a` and `b` are two names for the same numbers. Change one and you change both. This is the most common bug in this module's solvers.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = a                 # NOT a copy
b[0] = 99
print(a)              # a has changed too!
c = a.copy()          # a real copy
c[0] = -1
print(a, c)

**2.1** A function that 'steps' a state should not change the state you gave it. Fix `step` below so that `start` is unchanged after the call.

In [ ]:
def step(y, h):
    y[0] = y[0] + h * y[1]      # changes the caller's array!
    return y

start = np.array([1.0, 2.0])
new = step(start, 0.1)
print(start, new)

**Check:** `start` is still `[1. 2.]`; `new` is `[1.2 2. ]`.

## <span style="color:#B5121B">3. Indexing and slicing</span>

`x[i]` is one element (from 0; `x[-1]` is the last). `x[a:b]` is elements `a` up to, **not including**, `b`. `x[::2]` takes every second one.

In [ ]:
x = np.array([10, 11, 12, 13, 14, 15])
print(x[0], x[-1], x[2:5], x[::2])
print(x[1:] - x[:-1])          # each element minus the one before: differences of neighbours

**3.1** From `x`, take the first three values, and the last two.

In [ ]:
# 3.1

**Check:** `[10 11 12]` and `[14 15]`.

**3.2** Beat times `beats = np.array([0.0, 0.8, 1.7, 2.5, 3.4])`. The intervals are the differences of neighbours: compute them with slices (no loop), and check with `np.diff`.

In [ ]:
# 3.2

**Check:** `[0.8 0.9 0.8 0.9]` both ways (up to rounding).

**3.3** A central difference uses the neighbours on *both* sides: `(y[2:] - y[:-2]) / (2*h)`. With `t` and `y = t**2` from 1.1 (`h = 0.5`), compute it. What should the derivative of $t^2$ be?

In [ ]:
# 3.3

**Check:** exactly `2t` at the inner points: 1, 2, 3, …, 9.

## <span style="color:#B5121B">4. Boolean masks</span>

A comparison gives an array of `True`/`False` (a **mask**). Use it to select, and `.sum()` to count. Combine masks with `&` (and) and `|` (or), with brackets.

In [ ]:
temp = np.array([12.1, 13.4, -999.0, 14.0, 15.2, -999.0, 13.8])
bad = temp == -999.0
print(bad, bad.sum(), temp[~bad])          # ~ means 'not'
print(temp[(temp > 13) & (temp < 15)])

**4.1** How many readings are above 13.5? What is the mean of the real (not −999) readings?

In [ ]:
# 4.1

**Check:** 3 above 13.5; mean of the real readings 13.70.

**4.2** Replace the −999 values by `np.nan` (on a copy!), and use `np.nanmean`.

In [ ]:
# 4.2

**Check:** 13.70 again.

**4.3** Find the **positions** (indices) of the −999 values with `np.where`.

In [ ]:
# 4.3

**Check:** `[2 5]`.

## <span style="color:#B5121B">5. Loops that fill an array</span>

Step-by-step methods (Euler, RK4) fill an array one row at a time. Make the array first (`np.empty` or `np.zeros`), set the first value, then loop over the **number of steps** (an integer), not over a time that you keep adding to.

In [ ]:
N = 5
y = np.empty(N + 1)
y[0] = 1.0
for n in range(N):
    y[n + 1] = y[n] * 2          # each value from the one before
print(y)

**5.1** Fill an array with $y_{n+1} = y_n - h\,y_n$ (Euler for $dy/dt = -y$), $y_0 = 1$, $h = 0.1$, for 10 steps. Compare the last value with the exact $e^{-1}$.

In [ ]:
# 5.1

**Check:** 0.3487 against 0.3679.

**5.2** Why loop over an integer? Run `t = 0.0` then `while t < 1.0: t += 0.1` and count the steps. How many do you expect, and how many do you get?

In [ ]:
# 5.2

**Check:** 11 steps, not 10, ending at 1.0999…: 0.1 can't be stored exactly, so after ten additions `t` is 0.9999999999999999, still below 1. Use `N = round(t_end / h)` and `for n in range(N)`.

**5.3** A state with two numbers (angle and speed): fill a `(N+1, 2)` array, starting from `[1.0, 0.0]`, with the update `new = [old[0] + h*old[1], old[1] - h*old[0]]`, `h = 0.1`, 5 steps. Print the last row.

In [ ]:
# 5.3

**Check:** `[ 0.9005  -0.49001]`.

## <span style="color:#B5121B">6. Functions</span>

A function takes inputs, does one job, and **returns** the result. Write one for anything you do more than once. A docstring says what goes in and what comes out.

In [ ]:
def period(L, g=9.81):
    """Small-angle period (s) of a pendulum of length L (m)."""
    return 2 * np.pi * np.sqrt(L / g)

print(period(0.8), period(0.8, g=9.814))

**6.1** Write `mean_and_sem(x)` that returns two numbers: the mean and the standard error $s/\sqrt{N}$ (use `ddof=1`). Test it on `[1, 2, 3, 4, 5]`.

In [ ]:
# 6.1

**Check:** 3.0 and 0.707.

**6.2** A common mistake: `print` instead of `return`. What does `r = f(2)` give for the function below? Fix it.

In [ ]:
def f(x):
    print(x**2)

r = f(2)
print(r)

**Check:** the broken version prints 4 but `r` is `None`; the fixed version gives `r = 4`.

## <span style="color:#B5121B">7. Functions as inputs</span>

A solver, a root finder or `curve_fit` takes **your function** as an input and calls it. Pass the name, without brackets.

In [ ]:
def apply_twice(func, x):
    return func(func(x))

print(apply_twice(np.sqrt, 16.0))           # sqrt(sqrt(16)) = 2
print(apply_twice(lambda v: v + 1, 0))      # a lambda is a one-line function

**7.1** Write `euler(f, y0, h, N)` for one variable: `y[n+1] = y[n] + h*f(y[n])`, returning the array. Use it with `f = lambda y: -y` to repeat 5.1.

In [ ]:
# 7.1

**Check:** 0.3487, as in 5.1.

## <span style="color:#B5121B">8. Plots that say something</span>

Every figure needs axis labels with units, and a legend if there is more than one line. Plot data as points and models as lines.

In [ ]:
t = np.linspace(0, 10, 50)
data = np.exp(-t / 3) + 0.03 * np.random.default_rng(0).standard_normal(len(t))
plt.plot(t, data, "o", ms=3, label="measurements")
plt.plot(t, np.exp(-t / 3), label="model: $e^{-t/3}$")
plt.xlabel("Time (s)"); plt.ylabel("Signal (V)"); plt.legend(); plt.show()

**8.1** Plot the Euler result from 7.1 (as points) and the exact $e^{-t}$ (as a line) against time, with labels and a legend.

In [ ]:
# 8.1

**Check:** the points sit slightly below the line, and the gap grows with t.

## <span style="color:#B5121B">9. Just enough pandas</span>

A `DataFrame` is a table with named columns. Three things cover most of what you need: select rows with a mask, turn a column into a NumPy array, and make one row per group.

In [ ]:
df = pd.DataFrame({"person": ["a", "a", "b", "b", "b", "c"],
                   "group": ["young", "young", "old", "old", "old", "young"],
                   "rate": [62.0, 64.0, 55.0, 57.0, 56.0, 70.0]})
print(df[df.person == "b"])                 # rows for one person
r = df.loc[df.person == "b", "rate"].to_numpy()
print(r, r.mean())

**9.1** The mean rate for each **person** (one row per person) with `groupby`.

In [ ]:
# 9.1

**Check:** a 63.0, b 56.0, c 70.0.

**9.2** The same table, but built with a loop and a function that returns a dictionary: the pattern for Exercise 2. Add the person's group to each row.

In [ ]:
# 9.2

**Check:** three rows: a (young, 63.0, 2), b (old, 56.0, 3), c (young, 70.0, 1).

## Where next

When these feel comfortable, go back to the lab sections that matter for the current exercise. The **Check** lines there work the same way. Stuck on something here? Bring it to the lab, or ask on Teams.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>